# 5. LangGraph Basics

Move the agent and weather-tool loop into nodes and conditional edges.

In [ ]:
import os
import operator
import json
import time

from datetime import date, timedelta
from typing import Annotated, Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from openai import OpenAI
from pydantic import BaseModel
from learning_langgraph.weather import get_weather

In [ ]:
# Graph State
class AgentState(TypedDict):
    messages: Annotated[list[dict], operator.add]
    usage: dict
    tool_complete: bool


def route_after_agent(
        state: AgentState,
) -> Literal["weather", "__end__"]:
    
    last_message = state["messages"][-1]
    if last_message.get("tool_calls"):
        return "weather"

    return END

In [ ]:
class WeatherInput(BaseModel):
    location: str
    start_date: date
    end_date: date

In [ ]:
api_key = os.getenv("AWS_BEARER_TOKEN_BEDROCK")

if not api_key:
    raise RuntimeError("AWS_BEARER_TOKEN_BEDROCK is required")

region = os.getenv("BEDROCK_REGION", "ap-south-1")
model_id = "google.gemma-3-4b-it"

client = OpenAI(
    api_key=api_key,
    base_url=f"https://bedrock-mantle.{region}.api.aws/v1",
)

In [ ]:
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": (
            "Get a live weather forecast for a location "
            "and exact travel dates"
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "Destination city and country",
                },
                "start_date": {
                    "type": "string",
                    "format": "date",
                    "description": "First travel date in YYYY-MM-DD format",
                },
                "end_date": {
                    "type": "string",
                    "format": "date",
                    "description": "Last travel date in YYYY-MM-DD format",
                },
            },
            "required": [
                "location",
                "start_date",
                "end_date",
            ],
            "additionalProperties": False,
        },
    },
}

In [ ]:
def empty_usage() -> dict:
    return {
        "calls": 0,
        "input_tokens": 0,
        "output_tokens": 0,
        "total_tokens": 0,
        "latency_ms": 0,
    }

def update_usage(
    current: dict,
    response,
    latency_ms: int,
) -> dict:
    call_usage = response.usage

    return {
        "calls": current["calls"] + 1,
        "input_tokens": (
            current["input_tokens"]
            + call_usage.prompt_tokens
        ),
        "output_tokens": (
            current["output_tokens"]
            + call_usage.completion_tokens
        ),
        "total_tokens": (
            current["total_tokens"]
            + call_usage.total_tokens
        ),
        "latency_ms": (
            current["latency_ms"]
            + latency_ms
        ),
    }

In [ ]:
def assistant_message_to_dict(message) -> dict:
    result = {
        "role": "assistant",
        "content": message.content,
    }

    if message.tool_calls:
        result["tool_calls"] = [
            {
                "id": tool_call.id,
                "type": "function",
                "function": {
                    "name": tool_call.function.name,
                    "arguments": tool_call.function.arguments,
                },
            }
            for tool_call in message.tool_calls
        ]

    return result

In [ ]:
# Agent Node
def agent_node(state: AgentState) -> dict:
    request = {
        "model": model_id,
        "messages": state["messages"],
        "temperature": 0.2,
        "max_tokens": 500,
    }

    if not state["tool_complete"]:
        request["tools"] = [weather_tool]
        request["tool_choice"] = "required"

    start = time.perf_counter()
    response = client.chat.completions.create(**request)
    latency_ms = round((time.perf_counter() - start) * 1000)

    assistant_message = assistant_message_to_dict(response.choices[0].message)

    return {
        "messages": [assistant_message],
        "usage": update_usage(
            state["usage"],
            response,
            latency_ms,
        ),
    }

In [ ]:
#Weather Node
def weather_node(state: AgentState) -> dict:
    assistant_message = state["messages"][-1]
    tool_calls = assistant_message.get("tool_calls", [])

    if len(tool_calls) != 1 :
        raise ValueError("Expected exactly one tool call")

    tool_call = tool_calls[0]
    if tool_call["function"]["name"] != "get_weather":
        raise ValueError(f"Expected get_weather tool call. {tool_call['function']['name']}")

    weather_input = WeatherInput.model_validate_json(
        tool_call["function"]["arguments"]
    )

    weather_result = get_weather(
        location = weather_input.location,
        start_date = weather_input.start_date,
        end_date = weather_input.end_date,
    )

    tool_result_message = {
        "role":"user",
        "content": (
             "The get_weather tool returned this JSON:\n"
            f"{json.dumps(weather_result)}\n"
            "Use only this data to answer the original question."
        )
    }

    return {
        "messages": [tool_result_message],
        "tool_complete": True,
    }

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("agent", agent_node)
builder.add_node("weather", weather_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges(
    "agent",
    route_after_agent,
)
builder.add_edge("weather", "agent")

graph = builder.compile()

In [ ]:
trip_start = date.today() + timedelta(days=3)
trip_end = trip_start + timedelta(days=3)

initial_state = {
    "messages": [
        {
            "role": "system",
            "content": (
                "You are a personal travel decision assistant. "
                "Use the weather tool for weather facts. "
                "Do not invent weather data. "
                "Mention the weather data source."
            ),
        },
        {
            "role": "user",
            "content": (
                f"I may visit Denpasar, Bali, Indonesia "
                f"from {trip_start} to {trip_end}. "
                "I want beaches and outdoor activities. "
                "Is the weather suitable?"
            ),
        },
    ],
    "usage": empty_usage(),
    "tool_complete": False,
}

result = graph.invoke(
    initial_state,
    config={"recursion_limit": 5},
)

In [ ]:
print("Final answer:")
print(result["messages"][-1]["content"])

print("\nUsage:")
print(json.dumps(result["usage"], indent=2))

In [ ]:
for index, message in enumerate(result["messages"]):
    print(f"\nMessage {index}")
    print(json.dumps(message, indent=2))